# Arjun Listening Agent — Fine-Tuning Pipeline (Colab Pro)

Runs **SFT → (RM + PPO)** and **SFT → DPO** on the same base, then evaluates
all four checkpoints on the held-out conversation.

**Before running:** create a Google Drive folder `My Drive/CSE534_project/`
and upload to it:
- `arjun_10_conversations_evolving_tool_calls_labeled.docx`
- `parse_dataset.py`, `train_sft.py`, `train_reward_model.py`,
  `train_ppo.py`, `train_dpo.py`, `eval.py`

**Runtime:** *Runtime → Change runtime type → A100* (Pro). T4 also works
for the 3B model but PPO will take ~2–3× longer.


## 1. Verify GPU


In [ ]:
!nvidia-smi


## 2. Mount Drive and cd into the project folder


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/CSE534_project
!ls


## 3. Install dependencies

Pinned versions that play well together. The first install pulls Unsloth
and its xformers/triton stack — takes ~3–5 minutes.


In [ ]:
# Core training stack
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl==0.11.4' 'peft==0.13.2' 'accelerate==1.0.1' 'bitsandbytes==0.44.1'
!pip install -q 'transformers==4.45.2' datasets

# For parsing the docx
!pip install -q python-docx

# Sanity check
import torch, transformers, trl, peft
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'transformers {transformers.__version__}  trl {trl.__version__}  peft {peft.__version__}')


## 4. Parse the dataset

Holds out conversation #10 as the test set. Produces
`sft.jsonl`, `dpo.jsonl`, `test.jsonl`, `stats.json`.


In [ ]:
!python parse_dataset.py arjun_10_conversations_evolving_tool_calls_labeled.docx --holdout 10
!cat stats.json


Inspect a couple of training records:


In [ ]:
!head -n 2 sft.jsonl
print('---')
!head -n 2 dpo.jsonl
print('---')
!head -n 2 test.jsonl


## 5. Stage 1 — Supervised Fine-Tuning (shared baseline)

Both branches start from this checkpoint. ~10–15 min on A100, ~30 min on T4.


In [ ]:
!python train_sft.py


## 6A. RLHF branch — Reward Model

Trains the scalar reward head on the preference pairs (~5–10 min).


In [ ]:
!python train_reward_model.py


## 6B. RLHF branch — PPO

Rolls out responses, scores with the RM, updates with KL-constrained policy
gradient. ~30–60 min on A100. **This is the slow step.** Use Colab Pro's
background execution if you need to step away (Tools → Settings).


In [ ]:
!python train_ppo.py


## 7. DPO branch (single stage)

Same preference pairs, no reward model. ~10–15 min on A100.


In [ ]:
!python train_dpo.py


## 8. Evaluate all four models on the held-out conversation

Loads each checkpoint in turn (so peak VRAM stays low), generates predictions,
scores tool/importance accuracy and action precision/recall.


In [ ]:
!python eval.py \
    --base Qwen/Qwen2.5-3B-Instruct \
    --sft  ./qwen-arjun-sft-merged \
    --dpo  ./qwen-arjun-dpo-merged \
    --ppo  ./qwen-arjun-ppo-final \
    --test ./test.jsonl


## 9. Save the winning model back to Drive

Anything you saved under `/content/drive/MyDrive/CSE534_project/` already
persists — but the merged model directories are large. If you only want one,
zip it for easier download:


In [ ]:
# Pick whichever performed best in eval. Example: DPO winner.
!zip -qr qwen-arjun-dpo-merged.zip qwen-arjun-dpo-merged
!ls -lh qwen-arjun-dpo-merged.zip


## 10. (Optional) Convert the winner to GGUF for Ollama on your Mac

Run this **only** for the model you actually want to serve. Takes ~5 min.


In [ ]:
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./qwen-arjun-dpo-merged \
    --outfile qwen-arjun-dpo-q4.gguf --outtype q4_k_m
!ls -lh qwen-arjun-dpo-q4.gguf


Download `qwen-arjun-dpo-q4.gguf` from Drive to your Mac, then on the Mac:

```bash
cd ~/Documents/CSE534\ project
cat > Modelfile <<'EOF'
FROM ./qwen-arjun-dpo-q4.gguf
EOF
ollama create qwen-arjun -f Modelfile
```

Then change `MODEL_NAME` in `listening_agent.py` to `"qwen-arjun"` and
swap its server path to `arjun_mcp_server.py`.
